In [12]:
from pathlib import Path
import cv2
import pandas as pd
from ultralytics import YOLO
import torch

In [13]:
Path("../data/test.mp4").exists()

True

In [14]:
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print(f"Selected device: {DEVICE}")

Selected device: cuda


In [15]:

# -------- CONFIG --------
VIDEO_PATH = "../data/test.mp4"         
OUT_VIDEO = "../data/tracked_chickens_yolo.mp4"
OUT_TABLE = "../data/chicken_tracks_yolo.parquet"

CONF_THRESH = 0.25                      # detection confidence threshold
IOU_THRESH = 0.45                       # NMS IoU threshold
TRACKER = "../bytetrack.yaml"              # built-in tracker config
# DEVICE = "mps"                           # set to 0 for GPU if available, else None for CPU
# DEVICE = "cuda"

# Filter detections to these classes (set to None to keep all)
ALLOWED_CLASSES = {"bird"}              # YOLOv8 COCO label; chickens are usually "bird"

# -------- STABLE LABELS MANAGER --------
STABLE_NAMES = ["Chicken A", "Chicken B", "Chicken C"]

# Per-name state
name_state = {n: {"track_id": None, "last_pos": None, "last_seen": -1} for n in STABLE_NAMES}

# Map current tracker IDs -> friendly names
id2name = {}

# Tunable heuristics (adjust to your resolution and motion)
ASSIGN_DIST_THRESH = 400   # pixels; max distance to consider "same chicken"
MISS_TOLERANCE = 60        # frames; how long we keep a slot alive without observations

In [16]:

# -------- UTILITIES --------
def draw_label(img, label, x1, y1):
    """Draw a filled rectangle behind text for readability, positioned above the box."""
    font = cv2.FONT_HERSHEY_SIMPLEX
    scale = 0.6
    thickness = 1
    (w, h), _ = cv2.getTextSize(label, font, scale, thickness)
    top_left = (int(x1), int(y1) - h - 6)
    bottom_right = (int(x1) + w + 6, int(y1))
    cv2.rectangle(img, top_left, bottom_right, (0, 0, 0), -1)
    cv2.putText(img, label, (int(x1) + 3, int(y1) - 4), font, scale, (255, 255, 255), thickness, cv2.LINE_AA)


def assign_stable_name(track_id, cx, cy, frame_idx, used_names_this_frame):
    """Return a stable name for this detection and update state.
       If track_id changes, rebind to the nearest existing chicken name.
    """
    # 1) If we already know this track_id, reuse its friendly name.
    if track_id in id2name:
        name = id2name[track_id]
        name_state[name]["last_pos"] = (cx, cy)
        name_state[name]["last_seen"] = frame_idx
        name_state[name]["track_id"] = track_id
        used_names_this_frame.add(name)
        return name

    # 2) Try to match to nearest active name by position (not too old, not already used).
    nearest_name, nearest_dist = None, float("inf")
    for name, st in name_state.items():
        if st["last_pos"] is None:
            continue
        if frame_idx - st["last_seen"] > MISS_TOLERANCE:
            continue
        if name in used_names_this_frame:
            continue
        px, py = st["last_pos"]
        d = ((cx - px)**2 + (cy - py)**2) ** 0.5
        if d < nearest_dist:
            nearest_name, nearest_dist = name, d

    if nearest_name is not None and nearest_dist <= ASSIGN_DIST_THRESH:
        # Rebind the friendly name from old tracker ID to the new track_id.
        old_id = name_state[nearest_name]["track_id"]
        if old_id is not None and old_id in id2name:
            del id2name[old_id]
        if track_id != -1:  # don't persist -1 in the map
            id2name[track_id] = nearest_name
        name_state[nearest_name]["track_id"] = track_id if track_id != -1 else name_state[nearest_name]["track_id"]
        name_state[nearest_name]["last_pos"] = (cx, cy)
        name_state[nearest_name]["last_seen"] = frame_idx
        used_names_this_frame.add(nearest_name)
        return nearest_name

    # 3) Otherwise, grab an unused/stale slot.
    for name, st in name_state.items():
        if name in used_names_this_frame:
            continue
        if st["last_pos"] is None or frame_idx - st["last_seen"] > MISS_TOLERANCE:
            if track_id != -1:
                id2name[track_id] = name
                name_state[name]["track_id"] = track_id
            name_state[name]["last_pos"] = (cx, cy)
            name_state[name]["last_seen"] = frame_idx
            used_names_this_frame.add(name)
            return name

    # 4) Fallback: raw ID (rare; e.g., >3 detections or unusual frame)
    return f"Chicken {track_id}"

In [17]:

# -------- LOAD MODEL --------
# model = YOLO("yolov8s.pt")  
model = YOLO("yolo11x.pt")  
# model = YOLO("yolo11x-pose.pt")  

In [18]:

# Get video metadata for writer
cap_meta = cv2.VideoCapture(VIDEO_PATH)
fps = cap_meta.get(cv2.CAP_PROP_FPS)
fps = float(fps) if fps and fps > 0 else 30.0
width = int(cap_meta.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap_meta.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap_meta.release()

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(OUT_VIDEO, fourcc, fps, (width, height))

records = []  # will store per-detection per-frame rows
frame_idx = -1


In [19]:
# -------- TRACKING LOOP --------
# The generator yields a Result per frame with tracked boxes (boxes.id).
for result in model.track(
    source=VIDEO_PATH,
    stream=True,
    conf=CONF_THRESH,
    iou=IOU_THRESH,
    tracker=TRACKER,
    persist=True,             # keep tracker state across frames
    device=DEVICE
):
    
    used_names_this_frame = set()

    frame_idx += 1
    frame = result.orig_img.copy()  # BGR image

    if result.boxes is None or len(result.boxes) == 0:
        out.write(frame)
        continue

    boxes = result.boxes

    # Iterate detections for this frame

    for b in boxes:
        x1, y1, x2, y2 = b.xyxy[0].tolist()
        w_box = x2 - x1
        h_box = y2 - y1

        conf = float(b.conf[0]) if b.conf is not None else 0.0
        cls_id = int(b.cls[0]) if b.cls is not None else -1
        cls_name = model.names.get(cls_id, "unknown")

        if ALLOWED_CLASSES and cls_name not in ALLOWED_CLASSES:
            continue

        track_id = int(b.id[0]) if hasattr(b, "id") and b.id is not None else -1

        cx = (x1 + x2) / 2.0
        cy = (y1 + y2) / 2.0
        cx_norm = cx / width
        cy_norm = cy / height

        # --- Assign stable friendly name ---
        pretty_name = assign_stable_name(track_id, cx, cy, frame_idx, used_names_this_frame)

        # Draw
        color = (0, 255, 0)
        cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
        label = f"{pretty_name} conf={conf:.2f} id={track_id}"
        draw_label(frame, label, x1, y1)

        # Save row
        records.append({
            "frame": frame_idx,
            "track_id": track_id,
            "chicken_name": pretty_name,
            "class_id": cls_id,
            "class_name": cls_name,
            "confidence": conf,
            "x1": x1, "y1": y1, "x2": x2, "y2": y2,
            "width": w_box, "height": h_box,
            "cx": cx, "cy": cy,
            "cx_norm": cx_norm, "cy_norm": cy_norm
        })


    # Write the augmented frame
    out.write(frame)



video 1/1 (frame 1/733) /home/prince/proj/chicken-behaviour-classifier/notebook/../data/test.mp4: 544x640 1 bed, 12.7ms
video 1/1 (frame 2/733) /home/prince/proj/chicken-behaviour-classifier/notebook/../data/test.mp4: 544x640 1 bed, 10.4ms
video 1/1 (frame 3/733) /home/prince/proj/chicken-behaviour-classifier/notebook/../data/test.mp4: 544x640 1 bed, 10.4ms
video 1/1 (frame 4/733) /home/prince/proj/chicken-behaviour-classifier/notebook/../data/test.mp4: 544x640 1 bed, 10.4ms
video 1/1 (frame 5/733) /home/prince/proj/chicken-behaviour-classifier/notebook/../data/test.mp4: 544x640 1 bed, 10.4ms
video 1/1 (frame 6/733) /home/prince/proj/chicken-behaviour-classifier/notebook/../data/test.mp4: 544x640 1 bed, 10.4ms
video 1/1 (frame 7/733) /home/prince/proj/chicken-behaviour-classifier/notebook/../data/test.mp4: 544x640 1 bed, 10.4ms
video 1/1 (frame 8/733) /home/prince/proj/chicken-behaviour-classifier/notebook/../data/test.mp4: 544x640 1 bed, 10.4ms
video 1/1 (frame 9/733) /home/prince/pr

In [20]:

# -------- SAVE OUTPUTS --------
out.release()

df = pd.DataFrame.from_records(records)
# Helpful indexing for analysis:
df.sort_values(["track_id", "frame"], inplace=True)
df.to_parquet(OUT_TABLE, index=False)  # requires pyarrow

print(f"Saved video to: {OUT_VIDEO}")
print(f"Saved tracks to: {OUT_TABLE}")

Saved video to: ../data/tracked_chickens_yolo.mp4
Saved tracks to: ../data/chicken_tracks_yolo.parquet


In [21]:
df = pd.read_parquet("../data/chicken_tracks_yolo.parquet")

In [22]:
df

,frame,track_id,chicken_name,class_id,class_name,confidence,x1,y1,x2,y2,width,height,cx,cy,cx_norm,cy_norm
0,83,-1,Chicken A,14,bird,0.507599,436.733978,381.406464,625.673096,532.418945,188.939117,151.012482,531.203537,456.912704,0.754550,0.793251
1,84,-1,Chicken A,14,bird,0.317530,432.558014,391.509308,617.165833,532.801025,184.607819,141.291718,524.861923,462.155167,0.745543,0.802353
2,90,-1,Chicken A,14,bird,0.394273,532.839539,363.648621,581.268982,456.911835,48.429443,93.263214,557.054260,410.280228,0.791270,0.712292
3,98,-1,Chicken B,14,bird,0.303849,494.355621,295.824280,569.709534,410.174805,75.353912,114.350525,532.032578,352.999542,0.755728,0.612846
4,127,-1,Chicken B,14,bird,0.496871,205.677475,123.161804,347.134125,323.865753,141.456650,200.703949,276.405800,223.513779,0.392622,0.388045
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1139,728,211,Chicken A,14,bird,0.729151,94.634567,328.064941,196.809189,407.803925,102.174622,79.738983,145.721878,367.934433,0.206991,0.638775
1140,729,211,Chicken A,14,bird,0.763335,95.460930,329.667358,195.343216,406.885162,99.882286,77.217804,145.402073,368.276260,0.206537,0.639369
1141,730,211,Chicken A,14,bird,0.662749,94.881744,328.989746,194.394470,405.346252,99.512726,76.356506,144.638107,367.167999,0.205452,0.637444
1142,731,211,Chicken A,14,bird,0.761585,95.720428,328.785034,201.924728,409.550385,106.204300,80.765350,148.822578,369.167709,0.211396,0.640916


In [23]:
df.chicken_name.unique()

array(['Chicken A', 'Chicken B', 'Chicken C', 'Chicken -1', 'Chicken 15', 'Chicken 65', 'Chicken 66'], dtype=object)

In [24]:
df.chicken_name.value_counts()

chicken_name
Chicken B     439
Chicken C     352
Chicken A     337
Chicken 15     12
Chicken 66      2
Chicken -1      1
Chicken 65      1
Name: count, dtype: int64